In [ ]:
"""
Follow the tutorial found at https://sep.readthedocs.io/en/stable/tutorial.html, 
applying it to the data file you just downloaded. For best results (and for full credit) 
follow these suggestions: 

Gradually copy the code from the tutorial into your own jupyter notebook, run and debug 
until you successfully replicate the tutorial. 
Do not try to copy and run it all at once, it's not going to work! You need to debug as 
you go.
Use markdown to break the notebook into sections, as in the tutorial.
Finish your notebook by answering the following questions, using a combination of text 
and code directly in the notebook. 
(Terms like sources and fluxes will gradually become clear as you progress through the 
tutorial. You are not expected to fully understand all the details.) 
1) How many sources (stars) do you find in the data? Histogram their fluxes.
2) What are the mean, median, and standard deviation of the distribution of fluxes.
3) What is the largest outlier in the distribution; how many standard deviations is it
away from the mean?
"""

In [ ]:
import numpy as np
import sep

In [ ]:
# additional setup for reading the test image and displaying plots
from astropy.io import fits
import matplotlib.pyplot as plt
from matplotlib import rcParams

%matplotlib inline

rcParams['figure.figsize'] = [10., 8.]

In [ ]:
# read image into standard 2-d numpy array
data = fits.getdata("Desktop/hlsp_hudf12_hst_wfc3ir_udfmain_f105w_v1.0_drz.fits")

In [ ]:
data = data.astype(data.dtype.newbyteorder('='))

In [ ]:
# show the image
m, s = np.mean(data), np.std(data)
plt.imshow(data, interpolation='nearest', cmap='gray', vmin=m-s, vmax=m+s, origin='lower')
plt.colorbar();

In [ ]:
# measure a spatially varying background on the image
bkg = sep.Background(data)

In [ ]:
# get a "global" mean and noise of the image background:
print(bkg.globalback)
print(bkg.globalrms)

In [ ]:
# evaluate background as 2-d array, same size as original image
bkg_image = bkg.back()
# bkg_image = np.array(bkg) # equivalent to above

In [ ]:
# show the background
plt.imshow(bkg_image, interpolation='nearest', cmap='gray', origin='lower')
plt.colorbar();

In [ ]:
# evaluate the background noise as 2-d array, same size as original image
bkg_rms = bkg.rms()

In [ ]:
# show the background noise
plt.imshow(bkg_rms, interpolation='nearest', cmap='gray', origin='lower')
plt.colorbar();

In [ ]:
# subtract the background
data_sub = data - bkg

In [ ]:
objects = sep.extract(data_sub, 1.5, err=bkg.globalrms)

In [ ]:
# how many objects were detected
len(objects)

In [ ]:
from matplotlib.patches import Ellipse

# plot background-subtracted image
fig, ax = plt.subplots()
m, s = np.mean(data_sub), np.std(data_sub)
im = ax.imshow(data_sub, interpolation='nearest', cmap='gray',
               vmin=m-s, vmax=m+s, origin='lower')

# plot an ellipse for each object
for i in range(len(objects)):
    e = Ellipse(xy=(objects['x'][i], objects['y'][i]),
                width=6*objects['a'][i],
                height=6*objects['b'][i],
                angle=objects['theta'][i] * 180. / np.pi)
    e.set_facecolor('none')
    e.set_edgecolor('red')
    ax.add_artist(e)

In [ ]:
# available fields
objects.dtype.names

In [ ]:
flux, fluxerr, flag = sep.sum_circle(data_sub, objects['x'], objects['y'],
                                     3.0, err=bkg.globalrms, gain=1.0)

In [ ]:
# show the first 10 objects results:
for i in range(10):
    print("object {:d}: flux = {:f} +/- {:f}".format(i, flux[i], fluxerr[i]))

In this section, I answer the final project questions using the detected sources and their measured fluxes from the SEP tutorial.

In [ ]:
num_sources = len(objects)
print("Number of detected sources:", num_sources)

I detected 8640 sources according to the data.

The number of detected sources in the data is shown below.

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(flux, bins=100, range=(-0.1, 0.2))
plt.xlabel("Flux")
plt.ylabel("Number of Sources")
plt.title("Histogram of Source Fluxes (Zoomed)")
plt.show()

The histogram above shows the distribution of flux values for the detected sources.

In [ ]:
meanofflux = np.mean(flux)
medianofflux = np.median(flux)
stdofflux = np.std(flux)

print("Mean flux:", meanofflux)
print("Median flux:", medianofflux)
print("Standard deviation of flux:", stdofflux)

The mean, median, and standard deviation of the flux distribution are shown above.

In [ ]:
thelargestoutlier = np.max(flux)
outlierstdaway = (thelargestoutlier - meanofflux) / stdofflux

print("Largest outlier flux:", thelargestoutlier)
print("Standard deviations from the mean:", outlierstdaway)

The largest outlier in the flux distribution is the source with the maximum flux value. The output above shows its flux and how many standard deviations away it is from the mean.